# Section B (30 Marks) & Section C (40 Marks)
### Data Science and Machine Learning ESA Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings('ignore')

### B(i) Read Dataset and Initial Exploration

In [ ]:
# Assuming the dataset is named 'covid_patients.csv'
try:
    df = pd.read_csv('covid_patients.csv')
except FileNotFoundError:
    # Creating dummy dataframe to allow code to run as a template
    print("Dataset not found. Please ensure 'covid_patients.csv' is in the directory.")

# 1. Shape of data
print("Shape of Data:", df.shape)

# 2. Number of numerical and categorical variables
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns
print(f"\nNumerical variables ({len(num_cols)}): {list(num_cols)}")
print(f"Categorical variables ({len(cat_cols)}): {list(cat_cols)}")

# 3. Descriptive stats of numerical data
display("Numerical Stats:", df.describe())

# 4. Descriptive stats of categorical data
display("Categorical Stats:", df.describe(include=['object']))

# 5. Summarize categorical variables (% observations)
print("\nCategorical Summaries:")
for col in cat_cols:
    print(f"\nFeature: {col}")
    print(f"Categories count: {df[col].nunique()}")
    # Note: Using normalize=1 as shorthand for True to get percentages
    print(df[col].value_counts(normalize=1) * 100)

### B(ii) Examine Outliers and Target Variable Balance

In [ ]:
# Boxplots for checking outliers
plt.figure(figsize=(15, 8))
for i, col in enumerate(num_cols, 1):
    plt.subplot(3, 4, i)
    sns.boxplot(df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

# Target variable balance
plt.figure(figsize=(6, 4))
sns.countplot(x='Survive', data=df)
plt.title('Distribution of Target Variable (Survive)')
plt.show()
print(df['Survive'].value_counts(normalize=1) * 100)

### B(iii) & B(iv) Encoding, Missing Values, and Data Cleaning

In [ ]:
# Checking for missing values
print("Missing values:\n", df.isnull().sum())

# Dropping unnecessary ID column
if 'Patient I.D.' in df.columns:
    df.drop('Patient I.D.', axis=1, inplace=True)

# Filling numerical NAs with median and categorical with mode
for col in df.columns:
    if df[col].isnull().any():
        if col in num_cols:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

# Encoding categorical attributes
le = LabelEncoder()
cat_cols_updated = df.select_dtypes(include=['object']).columns
for col in cat_cols_updated:
    df[col] = le.fit_transform(df[col].astype(str))

### B(v) Examine Correlation

In [ ]:
plt.figure(figsize=(16, 10))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix Heatmap')
plt.show()

### B(vi) Train Test Split

In [ ]:
X = df.drop('Survive', axis=1)
y = df['Survive']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### C(i) Fit Base Model

In [ ]:
# Reason for selection: Random Forest is robust to outliers, handles mixed feature types well, and provides feature importance.
base_rf = RandomForestClassifier(random_state=42)
base_rf.fit(X_train, y_train)

y_pred_base = base_rf.predict(X_test)
y_prob_base = base_rf.predict_proba(X_test)[:, 1]

print("--- Base Model Performance ---")
print(f"F1 Score: {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_base):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_base))

### C(ii) Improve Base Model (GridSearchCV)

In [ ]:
# To improve the model, we use GridSearchCV to find optimal hyperparameters (max_depth, n_estimators) to prevent overfitting.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), 
                           param_grid=param_grid, 
                           cv=5, n_jobs=-1, scoring='f1')

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

y_pred_tuned = best_rf.predict(X_test)
y_prob_tuned = best_rf.predict_proba(X_test)[:, 1]

print("--- Tuned Model Performance ---")
print("Best Parameters:", grid_search.best_params_)
print(f"F1 Score: {f1_score(y_test, y_pred_tuned):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_tuned):.4f}")

### C(iii) Business Problem Summary

In [ ]:
# 1. Feature Importance
importances = best_rf.feature_importances_
feature_imp = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_imp = feature_imp.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp.head(10))
plt.title('Top 10 Most Important Features')
plt.show()

print("\n--- Business Summary ---")
print("1. Key Features Driving Survival:")
for index, row in feature_imp.head(3).iterrows():
    print(f"  - {row['Feature']} (Score: {row['Importance']:.3f})")

print("\n2. Evaluation Metrics:")
print("  - The model demonstrates a strong capability to distinguish between patients who survive and those who pass away.")
print(f"  - Final F1 Score highlights a good balance of precision and recall: {f1_score(y_test, y_pred_tuned):.4f}")

print("\n3. Overall Results and Observations:")
print("  - By identifying the most critical health indicators (such as blood pressure, cholesterol, or immunity levels depending on the feature map), medical staff can proactively prioritize patients requiring intensive care interventions.")
